In [4]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.6 MB/s eta 0:00:00


In [5]:
import os, pickle, itertools, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data   import DataLoader
from torch_geometric.nn     import ChebConv, global_mean_pool, global_max_pool
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

In [6]:
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Mounted at /content/drive
Using device: cuda


In [7]:
BASE_PATH  = '/content/drive/MyDrive/team3_xai_gnn'
DATA_DIR   = os.path.join(BASE_PATH, 'preprocessed')
OUTPUT_DIR = os.path.join(BASE_PATH, 'chebconv_results')
INNER_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'inner_results.pkl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# --- ChebGNN model -----------------------------------------------------------
#   - ChebConv αντί για custom attention layer
#   - Τα 15 DTI edge features συμπιέζονται σε 1 scalar weight (edge_encoder)
#     που περνάει ως edge_weight στο ChebConv
#   - Ο παράμετρος K ελέγχει πόσα hops βλέπει κάθε κόμβος (tunable)
#   - Readout: mean + max pooling

class ChebGNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden, K, dropout):
        super().__init__()

        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_in, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

        self.input_proj = nn.Linear(node_in, hidden)
        self.input_norm = nn.BatchNorm1d(hidden)

        self.conv1 = ChebConv(hidden, hidden, K=K)
        self.norm1 = nn.BatchNorm1d(hidden)

        self.conv2 = ChebConv(hidden, hidden, K=K)
        self.norm2 = nn.BatchNorm1d(hidden)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        edge_weight = self.edge_encoder(edge_attr).squeeze(-1)

        x = self.input_proj(x)
        x = self.input_norm(x)
        x = x.relu()

        x = self.conv1(x, edge_index, edge_weight)
        x = self.norm1(x)
        x = x.relu()
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_weight)
        x = self.norm2(x)
        x = x.relu()
        x = self.dropout(x)

        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=-1)

        return self.classifier(x)

In [9]:
# --- FocalLoss  -----------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss    = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            reduction='none'
        )
        pt         = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [10]:
# --- Helper functions  ---------------------------------
def get_labels(dataset):
    return np.array([d.y.item() for d in dataset])


def compute_metrics(labels, preds, probs):
    return {
        'balanced_acc':  balanced_accuracy_score(labels, preds),
        'accuracy':      accuracy_score(labels, preds),
        'recall_mci':    recall_score(labels, preds, pos_label=1, zero_division=0),
        'recall_cn':     recall_score(labels, preds, pos_label=0, zero_division=0),
        'precision_mci': precision_score(labels, preds, pos_label=1, zero_division=0),
        'precision_cn':  precision_score(labels, preds, pos_label=0, zero_division=0),
        'f1_mci':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'f1_cn':         f1_score(labels, preds, pos_label=0, zero_division=0),
        'f1_macro':      f1_score(labels, preds, average='macro', zero_division=0),
        'auc':           roc_auc_score(labels, probs[:, 1]) if len(np.unique(labels)) > 1 else float('nan'),
    }


def avg_metrics(metrics_list):
    keys = metrics_list[0].keys()
    return {k: float(np.mean([m[k] for m in metrics_list])) for k in keys}


def passes_filter(avg, thresholds):
    return all(avg.get(k, 0.0) >= v for k, v in thresholds.items())


# ίδια ακριβώς με Transformer: [1.0, mci_weight] — όχι compute_class_weight
def build_criterion(cfg, train_labels):
    weight = torch.tensor([1.0, cfg['mci_weight']], dtype=torch.float).to(device)
    return FocalLoss(gamma=cfg['gamma'], weight=weight)


def build_model_and_opt(cfg):
    model = ChebGNN(NODE_IN, EDGE_IN, HIDDEN, cfg['K'], DROPOUT).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    return model, opt


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_labels.append(batch.y.cpu().numpy())
        all_preds.append(probs.argmax(axis=1))
        all_probs.append(probs)
    return (
        np.concatenate(all_labels),
        np.concatenate(all_preds),
        np.concatenate(all_probs, axis=0),
    )


def train_with_early_stopping(model, opt, criterion, train_loader, val_loader):
    best_bal_acc   = -1.0
    best_state     = None
    best_metrics   = None
    patience_count = 0

    for epoch in range(EPOCHS):
        train_one_epoch(model, train_loader, criterion, opt)
        labels, preds, probs = evaluate(model, val_loader)
        metrics = compute_metrics(labels, preds, probs)

        if metrics['balanced_acc'] > best_bal_acc:
            best_bal_acc   = metrics['balanced_acc']
            best_state     = copy.deepcopy(model.state_dict())
            best_metrics   = metrics
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_metrics

In [11]:
# --- Fixed params --------------------------------------
NODE_IN  = 4
EDGE_IN  = 15
HIDDEN   = 32
DROPOUT  = 0.3
EPOCHS   = 50
PATIENCE = 15
BATCH    = 32
N_OUTER  = 5
N_INNER  = 3

gammas        = [1.0, 1.4, 1.8]
mci_weights   = [3.7, 4.2]
lrs           = [1e-3, 3e-3]
K_values      = [2, 3]
weight_decays = [5e-4, 1e-3]

CONFIGS = [
    {'mci_weight': mw, 'gamma': g, 'lr': lr, 'K': k, 'wd': wd}
    for mw, g, lr, k, wd in itertools.product(mci_weights, gammas, lrs, K_values, weight_decays)
]

print(f'Total configs: {len(CONFIGS)}')
for i, c in enumerate(CONFIGS):
    print(f'  [{i:02d}] {c}')

Total configs: 48
  [00] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'K': 2, 'wd': 0.0005}
  [01] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'K': 2, 'wd': 0.001}
  [02] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'K': 3, 'wd': 0.0005}
  [03] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.001, 'K': 3, 'wd': 0.001}
  [04] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'K': 2, 'wd': 0.0005}
  [05] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'K': 2, 'wd': 0.001}
  [06] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'K': 3, 'wd': 0.0005}
  [07] {'mci_weight': 3.7, 'gamma': 1.0, 'lr': 0.003, 'K': 3, 'wd': 0.001}
  [08] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'K': 2, 'wd': 0.0005}
  [09] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'K': 2, 'wd': 0.001}
  [10] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'K': 3, 'wd': 0.0005}
  [11] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.001, 'K': 3, 'wd': 0.001}
  [12] {'mci_weight': 3.7, 'gamma': 1.4, 'lr': 0.003, 'K': 2, 'wd': 0.0005}


In [12]:
# --- Data loading --------------------------------------
print('Loading data...')

try:
    outer_folds = []
    for k in range(1, N_OUTER + 1):
        outer_folds.append({
            'train': torch.load(os.path.join(DATA_DIR, f'fold_{k}_train.pt'), weights_only=False),
            'val':   torch.load(os.path.join(DATA_DIR, f'fold_{k}_val.pt'),   weights_only=False),
            'test':  torch.load(os.path.join(DATA_DIR, f'fold_{k}_test.pt'),  weights_only=False)
        })

    with open(os.path.join(DATA_DIR, 'inner_fold_ids.pkl'), 'rb') as f:
        inner_fold_ids = pickle.load(f)

    print('Done.')
    for k, fold in enumerate(outer_folds):
        print(f'  Outer fold {k+1}: train={len(fold["train"])}  val={len(fold["val"])}  test={len(fold["test"])}'
)
except OSError as e:
    if "Transport endpoint is not connected" in str(e):
        print("Error: Google Drive is likely disconnected. Please re-run the cell that mounts Google Drive (e.g., `drive.mount('/content/drive')`) and then try running this cell again.")
        raise RuntimeError("Google Drive disconnected, please remount.") from e
    else:
        raise

Loading data...
Done.
  Outer fold 1: train=431  val=77  test=127
  Outer fold 2: train=431  val=77  test=127
  Outer fold 3: train=431  val=77  test=127
  Outer fold 4: train=431  val=77  test=127
  Outer fold 5: train=431  val=77  test=127


In [14]:
# --- Inner loop: Hyperparameter tuning ---------------------------------------
if os.path.exists(INNER_RESULTS_PATH):
    with open(INNER_RESULTS_PATH, 'rb') as f:
        inner_results = pickle.load(f)
    print(f'Resuming - {len(inner_results)} runs already complete.')
else:
    inner_results = {}


def get_inner_split(outer_train, inner_fold_dict):
    inner_train = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_train']]
    inner_val   = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_val']]
    return inner_train, inner_val


outer_bar = tqdm(range(N_OUTER), desc='Outer folds', position=0)
for outer_idx in outer_bar:
    outer_train = outer_folds[outer_idx]['train']

    inner_bar = tqdm(range(N_INNER), desc=f'  Inner folds', position=1, leave=False)
    for inner_idx in inner_bar:
        inner_train, inner_val = get_inner_split(outer_train, inner_fold_ids[outer_idx][inner_idx])
        train_labels = get_labels(inner_train)
        train_loader = DataLoader(inner_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(inner_val,   batch_size=BATCH, shuffle=False)

        cfg_bar = tqdm(range(len(CONFIGS)), desc='    Configs', position=2, leave=False)
        for cfg_idx in cfg_bar:
            key = (outer_idx, inner_idx, cfg_idx)
            if key in inner_results:
                cfg_bar.set_postfix_str('skipped')
                continue

            cfg           = CONFIGS[cfg_idx]
            criterion     = build_criterion(cfg, train_labels)
            model, opt    = build_model_and_opt(cfg)
            _, metrics    = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

            inner_results[key] = metrics
            cfg_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                                recall_mci=f'{metrics["recall_mci"]:.3f}')

        # save after every inner fold
        with open(INNER_RESULTS_PATH, 'wb') as f:
            pickle.dump(inner_results, f)

print('Hyperparameter tuning completed.')

# average metrics per config across all 15 runs
inner_avg = {}
for cfg_idx in range(len(CONFIGS)):
    runs = [inner_results[(o, i, cfg_idx)] for o in range(N_OUTER) for i in range(N_INNER)]
    inner_avg[cfg_idx] = avg_metrics(runs)

print('\nInner-loop averaged metrics per config:')
df_inner = pd.DataFrame([
    {'config': i, **inner_avg[i], **CONFIGS[i]}
    for i in range(len(CONFIGS))
]).set_index('config')
print(df_inner.to_string())
df_inner.to_csv(os.path.join(OUTPUT_DIR, 'inner_avg_metrics.csv'))

Resuming - 720 runs already complete.


Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

...

  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

    Configs:   0%|          | 0/48 [00:00<?, ?it/s]

Hyperparameter tuning completed.

Inner-loop averaged metrics per config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr  K      wd
config                                                                                                                                                         
0           0.664309  0.601816    0.761789   0.566828       0.341620      0.853284  0.446755  0.650279  0.548517  0.694441         3.7    1.0  0.001  2  0.0005
1           0.666537  0.640391    0.710976   0.622098       0.340819      0.903642  0.450069  0.709452  0.579760  0.706879         3.7    1.0  0.001  2  0.0010
2           0.650120  0.558599    0.798296   0.501943       0.302685      0.847753  0.427361  0.617663  0.522512  0.688509         3.7    1.0  0.001  3  0.0005
3           0.675303  0.632929    0.745194   0.605413       0.342154      0.909625  0.456131  0.700497  0.578314  0.707451    

In [15]:
# --- Inner filter ---------------------------
INNER_FILTER = {
    'balanced_acc': 0.70,
    'recall_mci':   0.70,
    'recall_cn':    0.60,
}

surviving_configs = [
    i for i in range(len(CONFIGS))
    if passes_filter(inner_avg[i], INNER_FILTER)
]

print(f'Configs surviving inner filter: {len(surviving_configs)}/{len(CONFIGS)}')
for i in surviving_configs:
    print(f'  [{i:02d}] bal_acc={inner_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={inner_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={inner_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

Configs surviving inner filter: 4/48
  [36] bal_acc=0.714  recall_mci=0.775  recall_cn=0.653  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'K': 2, 'wd': 0.0005}
  [37] bal_acc=0.701  recall_mci=0.768  recall_cn=0.634  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'K': 2, 'wd': 0.001}
  [38] bal_acc=0.720  recall_mci=0.757  recall_cn=0.683  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'K': 3, 'wd': 0.0005}
  [39] bal_acc=0.707  recall_mci=0.771  recall_cn=0.643  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'K': 3, 'wd': 0.001}


In [16]:
# --- Outer loop: Training on full outer train + validation -------------------
CKPT_DIR               = os.path.join(OUTPUT_DIR, 'outer_checkpoints')
OUTER_VAL_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'outer_val_results.pkl')
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists(OUTER_VAL_RESULTS_PATH):
    with open(OUTER_VAL_RESULTS_PATH, 'rb') as f:
        outer_val_results = pickle.load(f)
    print(f'Resuming - {len(outer_val_results)} runs already complete.')
else:
    outer_val_results = {}


cfg_bar = tqdm(surviving_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc=f'  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        key = (cfg_idx, outer_idx)
        if key in outer_val_results:
            outer_bar.set_postfix_str('skipped')
            continue

        outer_train  = outer_folds[outer_idx]['train']
        outer_val    = outer_folds[outer_idx]['val']
        train_labels = get_labels(outer_train)
        train_loader = DataLoader(outer_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(outer_val,   batch_size=BATCH, shuffle=False)

        criterion      = build_criterion(cfg, train_labels)
        model, opt     = build_model_and_opt(cfg)
        model, metrics = train_with_early_stopping(model, opt, criterion, train_loader, val_loader)

        outer_val_results[key] = metrics
        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        torch.save(model.state_dict(), ckpt_path)

        with open(OUTER_VAL_RESULTS_PATH, 'wb') as f:
            pickle.dump(outer_val_results, f)

print('Training and validation completed.')

outer_val_avg = {
    i: avg_metrics([outer_val_results[(i, o)] for o in range(N_OUTER)])
    for i in surviving_configs
}

print('\nOuter-val averaged metrics per surviving config:')
df_val = pd.DataFrame([
    {'config': i, **outer_val_avg[i], **CONFIGS[i]}
    for i in surviving_configs
]).set_index('config')
print(df_val.to_string())
df_val.to_csv(os.path.join(OUTPUT_DIR, 'outer_val_avg_metrics.csv'))

Configs:   0%|          | 0/4 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

Training and validation completed.

Outer-val averaged metrics per surviving config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma     lr  K      wd
config                                                                                                                                                         
36          0.748593  0.714286    0.804167   0.693020       0.413968      0.943591  0.532658  0.786304  0.659481  0.783636         4.2    1.4  0.003  2  0.0005
37          0.709608  0.662338    0.788333   0.630883       0.359569      0.924575  0.488470  0.742725  0.615598  0.736269         4.2    1.4  0.003  2  0.0010
38          0.706581  0.672727    0.762500   0.650661       0.360352      0.922456  0.483574  0.753555  0.618565  0.740885         4.2    1.4  0.003  3  0.0005
39          0.728876  0.740260    0.710000   0.747753       0.456522      0.916879  0.532958  0.811784  0.672371  0

In [19]:
# --- Outer val filter -----------------------
OUTER_VAL_FILTER = {
    'balanced_acc': 0.70,
    'recall_mci':   0.70,
    'recall_cn':    0.70,
}

final_configs = [
    i for i in surviving_configs
    if passes_filter(outer_val_avg[i], OUTER_VAL_FILTER)
]

print(f'Configs surviving outer val filter: {len(final_configs)}/{len(surviving_configs)}')
for i in final_configs:
    print(f'  [{i:02d}] bal_acc={outer_val_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={outer_val_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={outer_val_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

if not final_configs:
    raise RuntimeError('No configs survived. Loosen OUTER_VAL_FILTER thresholds.')

Configs surviving outer val filter: 1/4
  [39] bal_acc=0.729  recall_mci=0.710  recall_cn=0.748  |  {'mci_weight': 4.2, 'gamma': 1.4, 'lr': 0.003, 'K': 3, 'wd': 0.001}


In [20]:
# --- Testing -----------------------------------------------------------------
test_results = {i: {} for i in final_configs}

cfg_bar = tqdm(final_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc='  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        model, _  = build_model_and_opt(cfg)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        test_loader          = DataLoader(outer_folds[outer_idx]['test'], batch_size=BATCH, shuffle=False)
        labels, preds, probs = evaluate(model, test_loader)
        metrics              = compute_metrics(labels, preds, probs)

        test_results[cfg_idx][outer_idx] = {
            'metrics': metrics,
            'probs':   probs,
            'preds':   preds,
            'labels':  labels,
        }

        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

# per-config average across outer folds
test_avg = {
    i: avg_metrics([test_results[i][o]['metrics'] for o in range(N_OUTER)])
    for i in final_configs
}

print('\nTest averaged metrics per final config:')
df_test = pd.DataFrame([
    {'config': i, **test_avg[i], **CONFIGS[i]}
    for i in final_configs
]).set_index('config')
print(df_test.to_string())
df_test.to_csv(os.path.join(OUTPUT_DIR, 'test_avg_metrics.csv'))

Configs:   0%|          | 0/1 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]


Test averaged metrics per final config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro      auc  mci_weight  gamma     lr  K     wd
config                                                                                                                                                       
39          0.652124   0.67874    0.614555   0.689693       0.345739      0.887186  0.432315  0.764968  0.598641  0.73926         4.2    1.4  0.003  3  0.001


In [21]:
# --- Ensemble (equal-weight + threshold 0.5) -
print('Ensemble results per outer fold:')
ensemble_results = {}

for outer_idx in range(N_OUTER):
    labels    = test_results[final_configs[0]][outer_idx]['labels']
    n         = len(labels)
    avg_probs = np.zeros((n, 2))

    for cfg_idx in final_configs:
        avg_probs += test_results[cfg_idx][outer_idx]['probs']
    avg_probs      /= len(final_configs)
    ensemble_preds  = avg_probs[:, 1] >= 0.5
    metrics         = compute_metrics(labels, ensemble_preds, avg_probs)

    ensemble_results[outer_idx] = {
        **metrics,
        'labels': labels,
        'preds':  ensemble_preds,
        'probs':  avg_probs,
    }

    print(f'  Outer fold {outer_idx+1}: '
          f'bal_acc={metrics["balanced_acc"]:.3f}  '
          f'recall_mci={metrics["recall_mci"]:.3f}  '
          f'recall_cn={metrics["recall_cn"]:.3f}  '
          f'auc={metrics["auc"]:.3f}')

ensemble_avg = avg_metrics([
    {k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}
    for o in range(N_OUTER)
])
print(f'\nEnsemble average across folds:')
for k, v in ensemble_avg.items():
    print(f'  {k}: {v:.4f}')

df_ensemble = pd.DataFrame([
    {'outer_fold': o+1, **{k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}}
    for o in range(N_OUTER)
] + [{'outer_fold': 'avg', **ensemble_avg}])
df_ensemble.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_results.csv'), index=False)
print(f'\nAll results saved to {OUTPUT_DIR}')

Ensemble results per outer fold:
  Outer fold 1: bal_acc=0.632  recall_mci=0.455  recall_cn=0.810  auc=0.763
  Outer fold 2: bal_acc=0.652  recall_mci=0.480  recall_cn=0.824  auc=0.642
  Outer fold 3: bal_acc=0.664  recall_mci=0.893  recall_cn=0.434  auc=0.824
  Outer fold 4: bal_acc=0.586  recall_mci=0.542  recall_cn=0.631  auc=0.678
  Outer fold 5: bal_acc=0.727  recall_mci=0.704  recall_cn=0.750  auc=0.789

Ensemble average across folds:
  balanced_acc: 0.6521
  accuracy: 0.6787
  recall_mci: 0.6146
  recall_cn: 0.6897
  precision_mci: 0.3457
  precision_cn: 0.8872
  f1_mci: 0.4323
  f1_cn: 0.7650
  f1_macro: 0.5986
  auc: 0.7393

All results saved to /content/drive/MyDrive/team3_xai_gnn/chebconv_results


In [22]:
# --- Confusion matrices -----------------------
for outer_idx in range(N_OUTER):
    print(f'---Outer Fold {outer_idx+1}---')

    for cfg_idx in final_configs:
        r  = test_results[cfg_idx][outer_idx]
        cm = confusion_matrix(r['labels'], r['preds'])
        c  = CONFIGS[cfg_idx]
        print(f'cfg{cfg_idx} (mw={c["mci_weight"]} g={c["gamma"]} lr={c["lr"]} K={c["K"]} wd={c["wd"]})')
        print(f'{"":10s}  Pred CN  Pred MCI')
        print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
        print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}\n')

    er = ensemble_results[outer_idx]
    cm = confusion_matrix(er['labels'], er['preds'])
    print(f'ENSEMBLE')
    print(f'{"":10s}  Pred CN  Pred MCI')
    print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
    print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}')
    print()

---Outer Fold 1---
cfg39 (mw=4.2 g=1.4 lr=0.003 K=3 wd=0.001)
            Pred CN  Pred MCI
  True CN     85       20
  True MCI    12       10

ENSEMBLE
            Pred CN  Pred MCI
  True CN     85       20
  True MCI    12       10

---Outer Fold 2---
cfg39 (mw=4.2 g=1.4 lr=0.003 K=3 wd=0.001)
            Pred CN  Pred MCI
  True CN     84       18
  True MCI    13       12

ENSEMBLE
            Pred CN  Pred MCI
  True CN     84       18
  True MCI    13       12

---Outer Fold 3---
cfg39 (mw=4.2 g=1.4 lr=0.003 K=3 wd=0.001)
            Pred CN  Pred MCI
  True CN     43       56
  True MCI     3       25

ENSEMBLE
            Pred CN  Pred MCI
  True CN     43       56
  True MCI     3       25

---Outer Fold 4---
cfg39 (mw=4.2 g=1.4 lr=0.003 K=3 wd=0.001)
            Pred CN  Pred MCI
  True CN     65       38
  True MCI    11       13

ENSEMBLE
            Pred CN  Pred MCI
  True CN     65       38
  True MCI    11       13

---Outer Fold 5---
cfg39 (mw=4.2 g=1.4 lr=0.003 K=3 